<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/3_3_Optimization_in_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part III. Evaluation Toolkit and Model Selection. 9. Optimization in Machine Learning

## 9.1. Градиентный спуск: полный пакетный, стохастический, мини-батч

Методы обучения, рассмотренные ранее — от линейной регрессии до SVM — сводятся к минимизации целевой функции: суммы потерь и регуляризатора. Для многих моделей аналитического решения не существует; вместо этого мы используем итерационные алгоритмы. Эффективность обучения — время сходимости и достижимая точность — напрямую зависит от выбранного оптимизатора. Эта глава раскрывает математику градиентных методов — от классического спуска до современных адаптивных алгоритмов, которые лежат в основе глубокого обучения.

---

#### 1. Постановка задачи и базовые понятия

Рассматривается задача безусловной минимизации дифференцируемой (необязательно выпуклой) функции $f : \mathbb{R}^d \to \mathbb{R}$:

$$
\mathbf{w}^* \in \arg\min_{\mathbf{w} \in \mathbb{R}^d} f(\mathbf{w}).
$$

Чаще всего $f$ представляет собой эмпирический риск: среднее значение функции потерь на обучающей выборке. Точное глобальное решение, как правило, недостижимо за конечное время, поэтому алгоритмы нацелены на нахождение $\epsilon$-стационарной точки, в которой $\|\nabla f(\mathbf{w})\| \le \epsilon$, либо на приближение к минимальному значению с заданной точностью.

Итерационный процесс **градиентного спуска** обновляет параметры по правилу

$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \eta_t \nabla f(\mathbf{w}_t),
$$

где $\eta_t > 0$ — **шаг** (learning rate). Градиент $\nabla f(\mathbf{w}_t)$ указывает направление наискорейшего возрастания функции; перемещение в противоположном направлении гарантирует локальное убывание при достаточно малом шаге. Величина шага контролирует, насколько далеко мы продвигаемся за одну итерацию.

---

#### 2. Полнопакетный градиентный спуск (Gradient Descent, GD)

Пусть $f(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^n L(\mathbf{w}; \mathbf{x}_i, y_i)$ — эмпирический риск. Тогда градиент также является средним:

$$
\nabla f(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^n \nabla L_i(\mathbf{w}).
$$

Каждая итерация GD требует вычисления всех $n$ градиентов, что обходится в $O(nd)$ операций. При больших $n$ это может быть неприемлемо дорого. Однако GD обладает сильными теоретическими гарантиями.

**Анализ сходимости.** Основные предположения — **$L$-гладкость** (градиент является липшицевым с константой $L$):

$$
\|\nabla f(\mathbf{w}) - \nabla f(\mathbf{v})\| \le L \|\mathbf{w} - \mathbf{v}\| \quad \forall \mathbf{w}, \mathbf{v}.
$$

Это свойство влечёт квадратичную верхнюю оценку:

$$
f(\mathbf{v}) \le f(\mathbf{w}) + \nabla f(\mathbf{w})^T(\mathbf{v} - \mathbf{w}) + \frac{L}{2}\|\mathbf{v} - \mathbf{w}\|^2.
$$

При $\eta_t = \eta \le 1/L$ и правиле обновления $\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \nabla f(\mathbf{w}_t)$ получаем гарантированное убывание:

$$
f(\mathbf{w}_{t+1}) \le f(\mathbf{w}_t) - \frac{\eta}{2}\|\nabla f(\mathbf{w}_t)\|^2.
$$

Отсюда следует, что метод сходится к стационарной точке, и среднее квадрата нормы градиента убывает как $O(1/t)$.

**Сильная выпуклость.** Функция $f$ называется $\mu$-сильно выпуклой, если

$$
f(\mathbf{v}) \ge f(\mathbf{w}) + \nabla f(\mathbf{w})^T(\mathbf{v} - \mathbf{w}) + \frac{\mu}{2}\|\mathbf{v} - \mathbf{w}\|^2 \quad \forall \mathbf{w}, \mathbf{v}.
$$

Для $L$-гладкой и $\mu$-сильно выпуклой функции GD с постоянным шагом $\eta = 1/L$ демонстрирует **линейную сходимость** (геометрическую прогрессию):

$$
\|\mathbf{w}_t - \mathbf{w}^*\|^2 \le \left(1 - \frac{\mu}{L}\right)^t \|\mathbf{w}_0 - \mathbf{w}^*\|^2.
$$

Число итераций до достижения точности $\epsilon$ составляет $O\bigl(\frac{L}{\mu} \log\frac{1}{\epsilon}\bigr)$. Если функция лишь выпукла (но не сильно), имеет место сублинейная скорость $f(\mathbf{w}_t) - f^* \le \frac{2L\|\mathbf{w}_0 - \mathbf{w}^*\|^2}{t}$.






**Углублённый взгляд: доказательство сходимости GD для выпуклых функций**

Пусть $f$ — выпуклая $L$-гладкая функция. Для $\eta = 1/L$ из квадратичной верхней оценки получаем:

$$
f(\mathbf{w}_{t+1}) \le f(\mathbf{w}_t) - \frac{1}{2L}\|\nabla f(\mathbf{w}_t)\|^2. \tag{1}
$$

Из выпуклости:
$$
f(\mathbf{w}_t) - f^* \le \nabla f(\mathbf{w}_t)^T(\mathbf{w}_t - \mathbf{w}^*). \tag{2}
$$

Теперь оценим изменение расстояния до оптимума:
$$
\begin{aligned}
\|\mathbf{w}_{t+1} - \mathbf{w}^*\|^2
&= \|\mathbf{w}_t - \eta \nabla f(\mathbf{w}_t) - \mathbf{w}^*\|^2 \\
&= \|\mathbf{w}_t - \mathbf{w}^*\|^2 - 2\eta \nabla f(\mathbf{w}_t)^T(\mathbf{w}_t - \mathbf{w}^*) + \eta^2 \|\nabla f(\mathbf{w}_t)\|^2.
\end{aligned}
$$

При $\eta = 1/L$, используя (2) для оценки скалярного произведения и неравенство (1) для оценки квадрата градиента, можно показать, что $\|\mathbf{w}_{t+1} - \mathbf{w}^*\|^2 \le \|\mathbf{w}_t - \mathbf{w}^*\|^2$. Тогда последовательность расстояний не возрастает, и, суммируя неравенства вида (1), получаем:

$$
\frac{1}{T}\sum_{t=0}^{T-1} \|\nabla f(\mathbf{w}_t)\|^2 \le \frac{2L(f(\mathbf{w}_0) - f^*)}{T}.
$$

Наконец, из выпуклости следует
$$
f(\bar{\mathbf{w}}_T) - f^* \le \frac{2L \|\mathbf{w}_0 - \mathbf{w}^*\|^2}{T},
$$
где $\bar{\mathbf{w}}_T = \frac{1}{T}\sum_{t=0}^{T-1} \mathbf{w}_t$ — усреднённый итерат.

Таким образом, GD гарантирует сублинейную скорость $O(1/T)$ для любых выпуклых $L$-гладких задач.



#### 3. Стохастический градиентный спуск (Stochastic Gradient Descent, SGD)

Когда число объектов $n$ огромно, вычисление полного градиента становится узким местом. SGD заменяет истинный градиент его **несмещённой стохастической оценкой**, построенной по одному случайно выбранному объекту $i$:

$$
\nabla f_i(\mathbf{w}) = \nabla L(\mathbf{w}; \mathbf{x}_i, y_i), \quad \mathbb{E}_i[\nabla f_i(\mathbf{w})] = \nabla f(\mathbf{w}).
$$

Обновление параметров принимает вид

$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \eta_t \nabla f_{i_t}(\mathbf{w}_t).
$$

Стоимость одной итерации падает до $O(d)$, однако вносится значительная дисперсия. Для сходимости необходимо правильно выбирать последовательность шагов: обычно $\eta_t \to 0$ со скоростью, обеспечивающей баланс между прогрессом и подавлением шума. Классические условия — $\sum \eta_t = \infty$ (чтобы алгоритм мог добраться до оптимума) и $\sum \eta_t^2 < \infty$ (чтобы шум не накапливался). Пример — $\eta_t = \eta_0 / \sqrt{t}$.

**Анализ сходимости SGD.** Для выпуклых $L$-гладких функций с ограниченной дисперсией градиента $\mathbb{E}\|\nabla f_i(\mathbf{w})\|^2 \le G^2$ и при убывающем шаге $\eta_t \sim 1/\sqrt{t}$ достигается скорость

$$
\mathbb{E}[f(\bar{\mathbf{w}}_T) - f^*] \le O\!\left(\frac{1}{\sqrt{T}}\right),
$$

где $\bar{\mathbf{w}}_T$ — усреднение итератов. При сильной выпуклости и правильно убывающем шаге можно получить $O(1/T)$. Константы, однако, зависят от дисперсии шума.

**Мини-батч SGD** — компромисс между GD и чистым SGD. Вместо одного объекта используют батч $B \subset \{1,\dots,n\}$ размера $b$:

$$
\nabla f_B(\mathbf{w}) = \frac{1}{b}\sum_{i \in B} \nabla f_i(\mathbf{w}).
$$

Дисперсия такой оценки уменьшается в $b$ раз по сравнению с одиночным объектом. Благодаря возможности векторизации (особенно на GPU) мини-батч SGD обеспечивает наилучшее использование вычислительных ресурсов и стал стандартом в глубоком обучении. Типичные размеры батча: 32, 64, 256.

---

#### 4. Практические аспекты и эвристики

- **Инициализация.** Для выпуклых задач начальное приближение может быть любым — сходимость гарантирована. В невыпуклых задачах (нейронные сети) начальные веса существенно влияют на итоговое качество; используются специальные схемы (Xavier, He).
- **Выбор шага $\eta$.** Слишком большой шаг вызывает расходимость или осцилляции; слишком маленький — медленную сходимость. Подбор обычно осуществляется по валидационной кривой.
- **Расписание шага.** Постоянный шаг в SGD приводит к осцилляциям вокруг оптимума. Убывающие схемы ($\eta_t = \eta_0 / (1 + \alpha t)$, косинусное затухание) стабилизируют сходимость. На практике часто применяют кусочно-постоянные спады.
- **Усреднение Поляка–Рупперта.** При постоянном шаге среднее итератов $\bar{\mathbf{w}}_T = \frac{1}{T}\sum_{t=1}^T \mathbf{w}_t$ сходится к оптимуму с оптимальной скоростью, сглаживая осцилляции. Это широко используется в SGD для выпуклых задач.

---

**Углублённый взгляд: почему SGD обобщает лучше, чем GD?**

Эмпирический факт: нейронные сети, обученные SGD с малым батчем, часто показывают лучшую обобщающую способность, чем обученные полным GD или большими батчами. Одно из объяснений: шум в оценке градиента действует как неявная регуляризация. Он помогает алгоритму избегать «острых» минимумов и находить широкие плоские области функции потерь, которые ассоциируются с лучшей обобщаемостью. Формально, добавление гауссовского шума к градиенту эквивалентно регуляризации, штрафующей норму гессиана, что поощряет сглаженные ландшафты.

---

#### 5. Сравнение GD, SGD и мини-батч

| Метод | Сложность итерации | Дисперсия градиента | Сходимость (итерации) | Сходимость (время) |
|-------|-------------------|---------------------|----------------------|--------------------|
| GD | $O(nd)$ | 0 | экспоненциальная (сильно выпуклые) | медленная для больших $n$ |
| SGD | $O(d)$ | высокая | сублинейная $O(1/\sqrt{t})$ | быстрая для больших $n$ |
| Mini-batch | $O(Bd)$ | $1/B$ | между GD и SGD | оптимальна на GPU |

**Рекомендации.**
- $n \le 10^4$: можно использовать GD — простота и стабильность.
- $10^4 < n < 10^6$: мини-батч SGD с $B = 64$–$256$ — использование векторизации.
- $n \ge 10^6$: SGD с малым батчем или чистый SGD, если память ограничена.
- Глубокое обучение: почти всегда мини-батч SGD с адаптивными расширениями (см. следующие разделы).





Запустите ячейку ниже. С помощью слайдеров вы можете менять шаг обучения, уровень шума в SGD и размер мини-батча. Обратите внимание:
- При большом шаге SGD расходится.
- Увеличение батча сглаживает траекторию, приближая её к GD.
- Шум помогает «выпрыгивать» из узких оврагов.

In [ ]:
# @title Траектории GD, SGD и Mini-batch SGD (интерактивный)
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

# Квадратичная функция с "овражной" структурой
def f(w):
    return 0.5 * (w[0]**2 + 10 * w[1]**2)

def grad(w):
    return np.array([w[0], 10 * w[1]])

def grad_stochastic(w, noise_level=2.0):
    return grad(w) + np.random.randn(2) * noise_level

def plot_trajectories(eta=0.05, noise=2.0, batch_size=16):
    np.random.seed(42)  # для воспроизводимости
    w0 = np.array([5.0, 3.0])
    T = 200

    # GD
    w_gd = w0.copy()
    path_gd = [w_gd.copy()]
    for t in range(T):
        w_gd = w_gd - eta * grad(w_gd)
        path_gd.append(w_gd.copy())

    # SGD (чистый)
    w_sgd = w0.copy()
    path_sgd = [w_sgd.copy()]
    for t in range(T):
        w_sgd = w_sgd - eta * grad_stochastic(w_sgd, noise)
        path_sgd.append(w_sgd.copy())

    # Mini-batch SGD
    w_mb = w0.copy()
    path_mb = [w_mb.copy()]
    for t in range(T):
        grad_mb = np.mean([grad_stochastic(w_mb, noise) for _ in range(batch_size)], axis=0)
        w_mb = w_mb - eta * grad_mb
        path_mb.append(w_mb.copy())

    path_gd = np.array(path_gd)
    path_sgd = np.array(path_sgd)
    path_mb = np.array(path_mb)

    # Визуализация
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Сетка для контуров
    x = np.linspace(-2, 6, 100)
    y = np.linspace(-3, 4, 100)
    X, Y = np.meshgrid(x, y)
    Z = f([X, Y])

    # Левый график: траектории
    ax1.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
    ax1.plot(path_gd[:,0], path_gd[:,1], 'o-', markersize=2, label='GD', color='blue')
    ax1.plot(path_sgd[:,0], path_sgd[:,1], 'o-', markersize=2, alpha=0.7, label='SGD', color='red')
    ax1.plot(path_mb[:,0], path_mb[:,1], 'o-', markersize=2, alpha=0.7,
             label=f'Mini-batch (b={batch_size})', color='green')
    ax1.scatter(0, 0, marker='*', s=200, color='gold', edgecolors='black', label='optimum')
    ax1.legend()
    ax1.set_title('Траектории на контурах функции потерь')
    ax1.set_xlabel('w1')
    ax1.set_ylabel('w2')
    ax1.grid(alpha=0.3)

    # Правый график: сходимость по функции
    ax2.plot([f(p) for p in path_gd], label='GD', color='blue')
    ax2.plot([f(p) for p in path_sgd], label='SGD', color='red', alpha=0.7)
    ax2.plot([f(p) for p in path_mb], label=f'Mini-batch (b={batch_size})', color='green', alpha=0.7)
    ax2.set_yscale('log')
    ax2.set_xlabel('Итерация')
    ax2.set_ylabel('f(w) (лог-шкала)')
    ax2.set_title('Сходимость по значению функции')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(plot_trajectories,
         eta=FloatSlider(value=0.05, min=0.005, max=0.2, step=0.005, description='Шаг η'),
         noise=FloatSlider(value=2.0, min=0.0, max=5.0, step=0.1, description='Шум SGD'),
         batch_size=IntSlider(value=16, min=1, max=128, step=1, description='Размер батча'));

### Вопросы для самопроверки

**Для начинающих**
1. Что такое градиентный спуск и как геометрически интерпретируется шаг в направлении антиградиента?
2. Зачем в стохастическом градиентном спуске шаг должен убывать со временем?
3. Чем SGD принципиально отличается от GD с точки зрения используемых данных на одной итерации?
4. Почему мини-батч SGD часто предпочтительнее как чистого SGD, так и полного GD?
5. Приведите пример задачи, где полнопакетный GD выгоднее SGD.

**Для профессионалов**
1. Докажите линейную сходимость GD для $L$-гладкой $\mu$-сильно выпуклой функции при $\eta = 1/L$.
2. Выведите условия сходимости SGD ($\sum \eta_t = \infty$, $\sum \eta_t^2 < \infty$) из анализа мартингалов или теоремы Роббинса–Монро.
3. Покажите, что дисперсия градиентной оценки в мини-батче размера $B$ равна $\frac{1}{B}$ дисперсии оценки по одному объекту (при независимом выборе объектов).
4. Сравните асимптотические скорости сходимости GD и SGD по числу обращений к данным (sample complexity). В каком смысле SGD «быстрее»?
5. Объясните регуляризующий эффект шума SGD, используя представление о неявной регуляризации через гессиан. Как это связано с плоскими минимумами?

### 9.2. Метод импульса (Momentum) и ускоренный градиент Нестерова (NAG)

Полнопакетный градиентный спуск и SGD не учитывают геометрию функции потерь за пределами локального градиента. Когда ландшафт обладает «овражной» структурой, обычный градиентный спуск осциллирует и сходится крайне медленно. Метод импульса и ускорение Нестерова решают эту проблему, вводя инерцию и «заглядывание вперёд».

---

#### 1. Проблема овражности (ravine problem)

Рассмотрим квадратичную функцию двух переменных с сильно различающейся кривизной:
$$
f(w_1, w_2) = w_1^2 + 100 w_2^2.
$$
Градиент $\nabla f = (2w_1, 200w_2)$ имеет большую компоненту вдоль оси $w_2$ и малую вдоль $w_1$. Градиентный спуск с постоянным шагом $\eta$, ограниченным обратной максимальной кривизной ($1/L \approx 1/200$), движется крошечными шагами вдоль пологого направления и резко осциллирует поперёк крутого склона. Траектория напоминает зигзаг, медленно сползающий ко дну оврага. Скорость сходимости определяется числом обусловленности $\kappa = L/\mu$; для данной функции $\kappa = 100$, и GD требует порядка $O(\kappa)$ итераций для достижения точности.

Импульсные методы подавляют поперечные колебания и накапливают движение вдоль дна оврага, имитируя физическую инерцию.

---

#### 2. Метод импульса (Momentum)

Метод импульса, или тяжёлого шарика (heavy ball), вводит вектор скорости $\mathbf{v}_t$, который экспоненциально сглаживает градиенты:

$$
\begin{aligned}
\mathbf{v}_{t+1} &= \beta \mathbf{v}_t + \nabla f(\mathbf{w}_t), \\
\mathbf{w}_{t+1} &= \mathbf{w}_t - \eta \mathbf{v}_{t+1}.
\end{aligned}
$$

Здесь $\beta \in [0,1)$ — коэффициент затухания импульса (обычно 0.9). Если $\beta = 0$, метод сводится к обычному GD. Рекуррентно раскрывая $\mathbf{v}_{t+1}$, получаем, что текущее обновление является экспоненциально взвешенным средним прошлых градиентов: компоненты, устойчиво указывающие в одном направлении, усиливаются, а знакопеременные (поперёк оврага) взаимно гасятся.

**Сходимость.** Для $L$-гладкой $\mu$-сильно выпуклой функции при оптимальных параметрах метод импульса обеспечивает линейную сходимость с коэффициентом, существенно меньшим, чем у GD:
$$
\|\mathbf{w}_t - \mathbf{w}^*\| \le \left( \frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1} \right)^t \|\mathbf{w}_0 - \mathbf{w}^*\|,
\qquad \kappa = \frac{L}{\mu}.
$$
В квадрате нормы это даёт $\bigl(1 - \Theta(1/\sqrt{\kappa})\bigr)^{2t}$, тогда как GD даёт $\bigl(1 - \Theta(1/\kappa)\bigr)^t$. Для больших $\kappa$ импульс сходится за $O(\sqrt{\kappa})$ итераций вместо $O(\kappa)$ у GD. Оптимальные параметры:
$$
\eta = \frac{2}{\sqrt{L} + \sqrt{\mu}}, \quad
\beta = \left( \frac{\sqrt{L} - \sqrt{\mu}}{\sqrt{L} + \sqrt{\mu}} \right)^2.
$$

---

**Углублённый взгляд: вывод оптимальных параметров импульса**

Применим метод импульса к квадратичной функции $f(\mathbf{w}) = \frac{1}{2}\mathbf{w}^T A \mathbf{w} - \mathbf{b}^T\mathbf{w}$ с симметричной положительно определённой матрицей $A$ ($L = \lambda_{\max}$, $\mu = \lambda_{\min}$). Итерация принимает вид линейной динамической системы:
$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \nabla f(\mathbf{w}_t) + \beta(\mathbf{w}_t - \mathbf{w}_{t-1}).
$$
В собственном базисе $A$ каждое направление эволюционирует независимо согласно скалярному разностному уравнению второго порядка. Сходимость характеризуется максимальным модулем корней характеристического уравнения $r^2 - (1+\beta - \eta \lambda) r + \beta = 0$. Чтобы минимизировать спектральный радиус при заданных $\lambda_{\min}, \lambda_{\max}$, параметры $\eta, \beta$ выбирают так, чтобы корни для крайних собственных значений были равны по модулю и комплексно сопряжены для внутренних. Это даёт приведённые выше выражения, а скорость сходимости определяется величиной $\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}$.

---

#### 3. Ускоренный градиент Нестерова (Nesterov Accelerated Gradient, NAG)

Нестеров предложил усовершенствование: вычислять градиент не в текущей точке, а в «предсказанной» позиции, куда импульс бы перенёс параметры без учёта нового градиента. Это позволяет «заглянуть вперёд» и скорректировать траекторию до того, как инерция уведёт слишком далеко.

Алгоритм NAG:
$$
\begin{aligned}
\mathbf{v}_{t+1} &= \beta \mathbf{v}_t + \nabla f(\mathbf{w}_t - \eta \beta \mathbf{v}_t), \\
\mathbf{w}_{t+1} &= \mathbf{w}_t - \eta \mathbf{v}_{t+1}.
\end{aligned}
$$
Здесь $\mathbf{w}_t - \eta\beta\mathbf{v}_t$ — экстраполированная точка, в которой вычисляется градиент. При $\beta = 0$ метод вновь вырождается в GD.

**Сходимость.** Для выпуклых $L$-гладких функций NAG обладает оптимальной скоростью первого порядка:
$$
f(\mathbf{w}_t) - f(\mathbf{w}^*) \le \frac{2L \|\mathbf{w}_0 - \mathbf{w}^*\|^2}{t^2}.
$$
Это значительно быстрее, чем $O(1/t)$ для GD или метода импульса. Нижняя оценка Нестерова утверждает, что любой метод, использующий только градиенты, не может сходиться быстрее $\Omega(1/t^2)$ на классе выпуклых $L$-гладких задач. Таким образом, NAG достигает неулучшаемой скорости.

---

**Углублённый взгляд: доказательство оптимальности Nesterov**

Идея Нестерова — построение *оценочной последовательности* (estimate sequence). Определим вспомогательную квадратичную функцию $\phi_t(\mathbf{w}) = \frac{\eta t^2}{2} \|\mathbf{w} - \mathbf{v}_t\|^2 + \text{const}$, которая мажорирует истинную функцию, и последовательно уменьшаем её. На каждом шаге гарантируется, что $f(\mathbf{w}_t) \le \min_{\mathbf{w}} \phi_t(\mathbf{w})$. Параметры подбираются так, чтобы обеспечить рекуррентное соотношение, приводящее к оценке $O(1/t^2)$. Полное изложение громоздко, но ключевой элемент — выбор коэффициента при $\mathbf{v}_t$ вида $\frac{t-1}{t+2}$ и адаптивный шаг вдоль разности двух последовательных точек. Именно этот механизм даёт квадратичное ускорение. Позднее было показано, что алгоритм NAG эквивалентен методу импульса с переменным $\beta_t$ и специфическим правилом «заглядывания».

---

#### 4. Геометрическая интерпретация

- **GD:** каждый шаг строго вдоль мгновенного антиградиента. На овражной функции траектория зигзагообразна, прогресс вдоль дна оврага медленный.
- **Momentum:** вектор скорости накапливает составляющую, направленную вдоль оврага, и гасит поперечные колебания. Траектория напоминает спуск тяжёлого шара, который минует дно, но быстро возвращается.
- **NAG:** «заглядывание» позволяет шару притормаживать заранее, не влетая на противоположный склон с избыточной скоростью. Траектория более прямая и быстрее стабилизируется около минимума.

На функции $x^2 + 100y^2$ GD требует сотни итераций, импульс — десятки, NAG — единицы (при правильном $\eta$).




Запустите ячейку ниже. На левом графике показаны траектории трёх методов на «овражной» функции $f(w) = w_1^2 + 100 w_2^2$, на правом — сходимость по значению функции. С помощью слайдеров можно менять шаг обучения и коэффициент импульса $\beta$.

In [ ]:
# @title Сравнение GD, Momentum и NAG (интерактивный)
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def f(w):
    return w[0]**2 + 100 * w[1]**2

def grad(w):
    return np.array([2*w[0], 200*w[1]])

def plot_optimizers(eta=0.02, beta=0.9):
    np.random.seed(42)
    w0 = np.array([5.0, 1.0])
    T = 150

    # GD
    w = w0.copy()
    path_gd = [w.copy()]
    for t in range(T):
        w = w - eta * grad(w)
        path_gd.append(w.copy())

    # Momentum
    w = w0.copy()
    v = np.zeros(2)
    path_mom = [w.copy()]
    for t in range(T):
        v = beta * v + grad(w)
        w = w - eta * v
        path_mom.append(w.copy())

    # NAG
    w = w0.copy()
    v = np.zeros(2)
    path_nag = [w.copy()]
    for t in range(T):
        w_lookahead = w - eta * beta * v
        v = beta * v + grad(w_lookahead)
        w = w - eta * v
        path_nag.append(w.copy())

    path_gd = np.array(path_gd)
    path_mom = np.array(path_mom)
    path_nag = np.array(path_nag)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Контуры
    x = np.linspace(-2, 6, 200)
    y = np.linspace(-2, 2, 200)
    X, Y = np.meshgrid(x, y)
    Z = f([X, Y])
    levels = np.logspace(-1, 3, 15)

    ax1.contour(X, Y, Z, levels=levels, cmap='viridis', alpha=0.6)
    ax1.plot(path_gd[:,0], path_gd[:,1], 'o-', markersize=2, label='GD', color='blue')
    ax1.plot(path_mom[:,0], path_mom[:,1], 'o-', markersize=2, label='Momentum', color='red')
    ax1.plot(path_nag[:,0], path_nag[:,1], 'o-', markersize=2, label='NAG', color='green')
    ax1.scatter(0, 0, marker='*', s=200, color='gold', edgecolors='black', label='optimum')
    ax1.legend()
    ax1.set_title('Траектории на контурах $w_1^2 + 100 w_2^2$')
    ax1.set_xlabel('$w_1$'); ax1.set_ylabel('$w_2$')
    ax1.grid(alpha=0.3)

    # Сходимость
    ax2.plot([f(p) for p in path_gd], label='GD', color='blue')
    ax2.plot([f(p) for p in path_mom], label='Momentum', color='red')
    ax2.plot([f(p) for p in path_nag], label='NAG', color='green')
    ax2.set_yscale('log')
    ax2.set_xlabel('Итерация')
    ax2.set_ylabel('$f(w)$ (лог-шкала)')
    ax2.set_title('Сходимость по значению функции')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(plot_optimizers,
         eta=FloatSlider(value=0.02, min=0.001, max=0.1, step=0.001, description='Шаг η'),
         beta=FloatSlider(value=0.9, min=0.0, max=0.99, step=0.01, description='Импульс β'));

#### 5. Практические рекомендации

- **Когда применять:** в задачах с плохо обусловленным гессианом (глубокие сети, логистическая регрессия с сильной корреляцией признаков), когда скорость GD неудовлетворительна.
- **Выбор $\beta$:** для импульса стандарт $\beta = 0.9$; для NAG часто используют $\beta = 0.9$–$0.99$, в глубоком обучении при обучении с cosine annealing $\beta$ может достигать $0.999$.
- **Связь с адаптивными методами:** Adam и его варианты объединяют идею импульса с адаптивной настройкой шага для каждого параметра (см. следующий раздел). Во многих фреймворках NAG реализован как опция `nesterov=True` в SGD с импульсом.

---

### Вопросы для самопроверки

**Для начинающих**
1. В чём заключается проблема овражности и как она влияет на обычный градиентный спуск?
2. Как метод импульса подавляет осцилляции поперёк оврага и ускоряет продвижение вдоль дна?
3. Чем алгоритм Нестерова отличается от классического импульса? Объясните идею «заглядывания вперёд».
4. Почему ускоренный градиент Нестерова сходится быстрее, чем обычный импульс для выпуклых функций?

**Для профессионалов**
1. Выведите оптимальные значения $\eta$ и $\beta$ для метода импульса на квадратичной функции, используя анализ характеристического уравнения и минимизацию спектрального радиуса.
2. Приведите набросок доказательства нижней оценки $\Omega(1/t^2)$ для методов первого порядка на классе выпуклых $L$-гладких функций. В чём суть построения «плохой» функции?
3. Проанализируйте влияние коэффициента $\beta$ на демпфирование осцилляций: как при $\beta \to 1$ изменяется динамика метода импульса? Когда это может привести к неустойчивости?
4. Существуют ли ситуации, в которых обычный импульс может превзойти NAG? Обоснуйте.
5. Обсудите связь метода NAG с методом сопряжённых градиентов. В каком смысле оба метода используют информацию о вторых производных без явного вычисления гессиана?

### 9.3. Адаптивные методы: AdaGrad, RMSprop, Adam

Методы градиентного спуска, рассмотренные ранее, используют единый скалярный шаг $\eta$ для всех параметров модели. На практике координаты вектора параметров $\mathbf{w}$ могут иметь совершенно разный масштаб и разную частоту обновления. Адаптивные оптимизаторы вводят индивидуальную скорость обучения для каждого параметра, опираясь на историю его градиентов. Это особенно важно при работе с разреженными данными и глубокими нейронными сетями.

---

#### 1. Проблема единого шага для всех параметров

Классический SGD обновляет все компоненты $\mathbf{w}$ с одинаковым шагом $\eta$. Недостатки такого подхода проявляются в двух типичных ситуациях.

- **Разреженные данные.** Если признаки закодированы one-hot и имеют миллионы размерностей, большинство признаков на каждой итерации принимают нулевые значения. Их градиенты равны нулю, тогда как градиенты частых признаков обновляются регулярно. При едином шаге редкие признаки почти не обучаются; чтобы они получили значимое обновление, требуется непропорционально много эпох.
- **Разный масштаб параметров.** В глубоких сетях разные слои обладают различной чувствительностью функции потерь. Общий шаг, приемлемый для одного слоя, может вызывать расходимость или, наоборот, чрезмерно медленное обучение в другом. Ручная настройка покоординатных шагов невозможна при миллионах параметров.

Идея адаптивных методов — накапливать информацию о величине градиентов вдоль каждой координаты и автоматически масштабировать шаг: для параметров с большими и частыми градиентами шаг уменьшается, для редких и малых — остаётся большим.

---

#### 2. AdaGrad (Adaptive Gradient)

AdaGrad независимо масштабирует шаг для каждого параметра $w_j$ обратно пропорционально квадратному корню из суммы квадратов прошлых градиентов. Пусть $g_{\tau,j} = \frac{\partial f}{\partial w_j}(\mathbf{w}_\tau)$ — частная производная на итерации $\tau$. Алгоритм накапливает диагональную матрицу

$$
G_{t, jj} = \sum_{\tau=1}^{t} g_{\tau, j}^2,
$$

а обновление имеет вид

$$
w_{t+1, j} = w_{t, j} - \frac{\eta}{\sqrt{G_{t, jj} + \epsilon}} \, g_{t, j},
$$

где $\eta$ — глобальный шаг, $\epsilon \sim 10^{-8}$ для численной стабильности.

**Свойства:**
- Частые признаки накапливают большую сумму квадратов градиентов, их эффективный шаг $\eta / \sqrt{G_{t, jj}}$ быстро уменьшается. Это автоматически снижает темп обучения для параметров, уже получивших много обновлений.
- Редкие признаки, наоборот, сохраняют большой шаг, что позволяет модели быстро адаптироваться к новой информации о них. Благодаря этому AdaGrad особенно хорош для разреженных данных — мешка слов, рекомендательных систем с категориальными признаками.
- Для выпуклых задач AdaGrad обладает строгими гарантиями сходимости.

**Недостаток:** $G_{t, jj}$ монотонно возрастает, поэтому эффективный шаг стремится к нулю. На плотных данных обучение может прекратиться задолго до достижения минимума. Это ограничивает применимость AdaGrad в глубоких невыпуклых задачах.

---

**Углублённый взгляд: сходимость AdaGrad для выпуклых функций**

Дучи и соавторы (2011) показали, что для выпуклых липшицевых функций AdaGrad достигает оценки

$$
\mathbb{E}\bigl[f(\bar{\mathbf{w}}_T) - f(\mathbf{w}^*)\bigr] \le O\!\left( \frac{\mathrm{tr}(G_T^{1/2})}{T} \right),
$$

где $\bar{\mathbf{w}}_T$ — среднее итератов. Величина $\mathrm{tr}(G_T^{1/2})$ тем меньше, чем более разрежены данные, — в этом случае AdaGrad значительно выигрывает у обычного SGD. Для плотных данных с равномерным распределением градиентов $\mathrm{tr}(G_T^{1/2}) = O(\sqrt{dT})$, и скорость остаётся $O(1/\sqrt{T})$.

---

#### 3. RMSprop (Root Mean Square Propagation)

Чтобы избежать неограниченного затухания шага, RMSprop заменяет полную сумму квадратов градиентов на экспоненциально взвешенное скользящее среднее. Обозначим через $g_{t}$ вектор градиента, все операции ниже покомпонентные. Алгоритм:

$$
\begin{aligned}
v_t &= \beta v_{t-1} + (1-\beta) g_t^2, \\
w_{t+1} &= w_t - \frac{\eta}{\sqrt{v_t + \epsilon}} \, g_t.
\end{aligned}
$$

Здесь $\beta$ — коэффициент затухания (обычно 0.9), $\epsilon$ — стабилизирующая добавка. Скользящее среднее $v_t$ отражает текущую «энергию» градиента и не растёт неограниченно.

**Свойства:**
- Шаг остаётся адаптивным, но больше не стремится к нулю — обучение может продолжаться сколь угодно долго.
- Метод хорошо зарекомендовал себя в задачах с нестационарной статистикой, таких как обучение рекуррентных нейронных сетей (RNN) и онлайн-обучение.
- RMSprop не имеет строгих доказательств сходимости для выпуклых функций, но его эмпирический успех огромен. Именно он стал основой для Adam.

---

#### 4. Adam (Adaptive Moment Estimation)

Adam объединяет две идеи: метод импульса (экспоненциальное среднее градиентов — первый момент) и RMSprop (экспоненциальное среднее квадратов градиентов — второй момент). Алгоритм поддерживает два состояния, $m_t$ и $v_t$, инициализированных нулями:

$$
\begin{aligned}
m_t &= \beta_1 m_{t-1} + (1-\beta_1) g_t \quad &\text{(оценка первого момента)}, \\
v_t &= \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \quad &\text{(оценка второго момента)}.
\end{aligned}
$$

Поскольку $m_0 = 0$ и $v_0 = 0$, на начальных итерациях эти оценки смещены к нулю. Для коррекции смещения вводятся нормированные величины:

$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \qquad
\hat{v}_t = \frac{v_t}{1 - \beta_2^t}.
$$

Обновление параметров выполняется по правилу:

$$
w_{t+1} = w_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \, \hat{m}_t.
$$

Рекомендованные значения: $\eta = 0.001$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$.

**Интуиция.** $\hat{m}_t$ играет роль «скорости» с памятью (импульс), а $\hat{v}_t$ адаптивно масштабирует шаг для каждого параметра. Комбинация позволяет быстро двигаться в направлениях с малым и устойчивым градиентом и тормозить при большой неопределённости.

**Свойства:**
- Adam — один из самых робастных оптимизаторов: он работает почти «из коробки» на широком спектре задач глубокого обучения (CNN, RNN, трансформеры).
- Совмещает преимущества адаптивного шага RMSprop и ускоряющих свойств импульса.
- Менее требователен к подбору гиперпараметров, чем SGD с моментом.

**Недостатки:**
- Reddi и соавторы (2018) показали, что Adam может расходиться даже на простых выпуклых задачах из-за недостаточно быстрого убывания $v_t$.
- В ряде случаев SGD с моментом достигает лучшей обобщающей способности: Adam иногда предпочитает «острые» минимумы.
- Существуют модификации (AMSGrad, AdamW), исправляющие некоторые из этих проблем.

---

**Углублённый взгляд: доказательство коррекции смещения в Adam**

Покажем, что $\hat{m}_t$ является несмещённой оценкой истинного первого момента $\mathbb{E}[g_t]$ при стационарном распределении градиентов. Раскроем рекурсию:

$$
m_t = (1-\beta_1) \sum_{i=1}^{t} \beta_1^{t-i} g_i.
$$

Математическое ожидание (при фиксированной истории) равно

$$
\mathbb{E}[m_t] = (1-\beta_1) \sum_{i=1}^{t} \beta_1^{t-i} \mathbb{E}[g_i].
$$

Если распределение градиентов стационарно, $\mathbb{E}[g_i] = \mu$ постоянно, и

$$
\mathbb{E}[m_t] = \mu (1-\beta_1) \sum_{i=1}^{t} \beta_1^{t-i} = \mu (1-\beta_1) \frac{1-\beta_1^t}{1-\beta_1} = \mu (1 - \beta_1^t).
$$

Следовательно, $\mathbb{E}[m_t / (1-\beta_1^t)] = \mu$, то есть $\hat{m}_t$ несмещена. Аналогично для второго момента. Без коррекции первые шаги имели бы значительно заниженную амплитуду, замедляя обучение.

---

#### 5. Сравнение и рекомендации

- **AdaGrad.** Идеален для разреженных данных (классификация текстов, рекомендательные системы с категориальными признаками), где редкие признаки должны получать более крупные обновления. В глубоких сетях применяется редко из-за затухания шага.
- **RMSprop.** Предназначен для нестационарных и онлайн-задач; исторически ключевой оптимизатор для рекуррентных сетей. Полезен, когда нужно долгое обучение без остановки.
- **Adam.** Универсальный оптимизатор для глубокого обучения. Рекомендуется в качестве первого выбора, когда нет ресурсов на тщательный подбор гиперпараметров SGD.
- **SGD с моментом.** Часто предпочтителен, когда важна предельная обобщающая способность (соревнования, production) и есть время на подбор расписания шага. При малых датасетах ($n \lesssim 10^5$) SGD часто оказывается не хуже Adam, а его динамика лучше изучена теоретически.

**Современные модификации:**
- **AdamW** — разделяет адаптивный шаг и весовой распад (weight decay), что даёт лучшую обобщающую способность.
- **AMSGrad** — гарантирует невозрастание $v_t$, решая проблему потенциальной расходимости Adam.
- **Nadam** — добавляет нестеровское «заглядывание» в Adam.

Практическое правило: начинайте с Adam (или AdamW), а при необходимости выжимать последние проценты качества переключайтесь на SGD с моментом и тщательно подобранным расписанием.



Запустите ячейку ниже. Вы увидите траектории SGD, AdaGrad, RMSprop и Adam на овражной функции $f(w) = w_1^2 + 100 w_2^2$. С помощью слайдеров можно менять глобальный шаг $\eta$ и сравнивать, как быстро каждый метод достигает минимума.

In [ ]:
# @title Сравнение SGD, AdaGrad, RMSprop и Adam (интерактивный)
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def f(w):
    return w[0]**2 + 100 * w[1]**2

def grad(w):
    return np.array([2*w[0], 200*w[1]])

def plot_adaptive(eta=0.1, beta2=0.99):
    np.random.seed(42)
    w0 = np.array([5.0, 1.0])
    T = 200
    eps = 1e-8

    # SGD
    w = w0.copy()
    path_sgd = [w.copy()]
    for t in range(T):
        w = w - eta * grad(w)
        path_sgd.append(w.copy())

    # AdaGrad
    w = w0.copy()
    G = np.zeros(2)
    path_adagrad = [w.copy()]
    for t in range(T):
        g = grad(w)
        G += g**2
        w = w - eta / (np.sqrt(G) + eps) * g
        path_adagrad.append(w.copy())

    # RMSprop
    w = w0.copy()
    v = np.zeros(2)
    path_rmsprop = [w.copy()]
    for t in range(T):
        g = grad(w)
        v = beta2 * v + (1 - beta2) * g**2
        w = w - eta / (np.sqrt(v) + eps) * g
        path_rmsprop.append(w.copy())

    # Adam
    w = w0.copy()
    m = np.zeros(2)
    v = np.zeros(2)
    beta1 = 0.9
    path_adam = [w.copy()]
    for t in range(1, T+1):
        g = grad(w)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        w = w - eta / (np.sqrt(v_hat) + eps) * m_hat
        path_adam.append(w.copy())

    paths = {
        'SGD': np.array(path_sgd),
        'AdaGrad': np.array(path_adagrad),
        'RMSprop': np.array(path_rmsprop),
        'Adam': np.array(path_adam)
    }
    colors = {'SGD': 'blue', 'AdaGrad': 'orange', 'RMSprop': 'red', 'Adam': 'green'}

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Контуры
    x = np.linspace(-2, 6, 200)
    y = np.linspace(-2, 2, 200)
    X, Y = np.meshgrid(x, y)
    Z = f([X, Y])
    levels = np.logspace(-1, 3, 15)

    ax1.contour(X, Y, Z, levels=levels, cmap='viridis', alpha=0.6)
    for name, path in paths.items():
        ax1.plot(path[:,0], path[:,1], 'o-', markersize=2, label=name, color=colors[name])
    ax1.scatter(0, 0, marker='*', s=200, color='gold', edgecolors='black', label='optimum')
    ax1.legend()
    ax1.set_title('Траектории на $w_1^2 + 100 w_2^2$')
    ax1.set_xlabel('$w_1$'); ax1.set_ylabel('$w_2$')
    ax1.grid(alpha=0.3)

    # Сходимость
    for name, path in paths.items():
        ax2.plot([f(p) for p in path], label=name, color=colors[name])
    ax2.set_yscale('log')
    ax2.set_xlabel('Итерация')
    ax2.set_ylabel('$f(w)$ (лог-шкала)')
    ax2.set_title('Сходимость по значению функции')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(plot_adaptive,
         eta=FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description='Шаг η'),
         beta2=FloatSlider(value=0.99, min=0.8, max=0.999, step=0.001, description='β₂ (RMSprop/Adam)'));



### Вопросы для самопроверки

**Для начинающих**
1. Почему использование одного и того же шага $\eta$ для всех параметров может быть неэффективным?
2. Как AdaGrad масштабирует шаг? Почему он хорошо работает для разреженных данных?
3. Какую проблему AdaGrad решает RMSprop, и как именно (технически) он это делает?
4. Из каких компонент состоит Adam? Зачем нужна коррекция смещения в начальных итерациях?

**Для профессионалов**
1. Выведите условие, при котором шаг AdaGrad монотонно убывает, и объясните, почему это приводит к преждевременной остановке обучения на плотных данных.
2. Проанализируйте экспоненциальное скользящее среднее в RMSprop: как выбор $\beta$ влияет на «память» метода и его способность адаптироваться к изменяющейся кривизне?
3. Докажите несмещённость $\hat{m}_t$ в Adam при стационарном распределении градиентов. Почему без коррекции смещения обучение на первых шагах было бы замедлено?
4. Сравните обобщающую способность Adam и SGD с моментом. Объясните, как адаптивный шаг может приводить к выбору более «острых» минимумов и почему это нежелательно.
5. Приведите контрпример (логику), демонстрирующий возможность расходимости Adam на выпуклой задаче, и объясните, как AMSGrad устраняет эту проблему.

### 9.4. Сравнение оптимизаторов на задачах ML и связь с валидацией

Теоретический анализ сходимости и интуиция, развитая в предыдущих разделах, нуждаются в экспериментальной проверке. Здесь мы на синтетических данных сопоставим поведение градиентных методов, выявим их практические сильные и слабые стороны, а также покажем глубокую связь между оптимизацией и процедурами выбора модели, включая раннюю остановку.

Запустите ячейку ниже. Она содержит всю реализацию: логистическую регрессию с нуля, классы GD, SGD, Momentum и Adam, а также интерактивные слайдеры для выбора оптимизатора и learning rate. Вы увидите кривые сходимости на обучающей и валидационной выборках.

In [ ]:
# @title Эксперимент 1: логистическая регрессия — сравнение оптимизаторов (интерактивный)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from ipywidgets import interact, Dropdown, FloatLogSlider

# ================================================
# Генерация данных
# ================================================
X, y = make_classification(n_samples=10000, n_features=20, n_informative=15,
                           n_redundant=3, n_clusters_per_class=2,
                           weights=[0.6, 0.4], flip_y=0.05, random_state=42)
X = np.c_[X, np.ones(X.shape[0])]
n_features = X.shape[1]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ================================================
# Функции потерь и градиента
# ================================================
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_loss_grad(w, Xb, yb, lam):
    z = Xb @ w
    p = sigmoid(z)
    loss = -np.mean(yb * np.log(p + 1e-15) + (1 - yb) * np.log(1 - p + 1e-15)) + 0.5 * lam * np.dot(w, w)
    grad = (Xb.T @ (p - yb)) / Xb.shape[0] + lam * w
    return loss, grad

# ================================================
# Классы оптимизаторов
# ================================================
class GDOptimizer:
    def __init__(self, lr): self.lr = lr
    def update(self, w, grad): return w - self.lr * grad

class SGDOptimizer:
    def __init__(self, lr): self.lr = lr
    def update(self, w, grad): return w - self.lr * grad

class MomentumOptimizer:
    def __init__(self, lr, beta=0.9):
        self.lr, self.beta, self.v = lr, beta, None
    def update(self, w, grad):
        if self.v is None: self.v = np.zeros_like(w)
        self.v = self.beta * self.v + grad
        return w - self.lr * self.v

class AdamOptimizer:
    def __init__(self, lr, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self.m, self.v, self.t = None, None, 0
    def update(self, w, grad):
        if self.m is None:
            self.m = np.zeros_like(w)
            self.v = np.zeros_like(w)
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1 - self.beta2) * grad * grad
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)
        return w - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

# ================================================
# Функция обучения
# ================================================
def train_one_epoch(w, X, y, opt, lam, batch_size='full', shuffle=True):
    n = X.shape[0]
    if batch_size == 'full':
        loss, grad = compute_loss_grad(w, X, y, lam)
        w = opt.update(w, grad)
        return w, loss
    else:
        if shuffle:
            idx = np.random.permutation(n)
            X, y = X[idx], y[idx]
        total_loss = 0
        for i in range(0, n, batch_size):
            Xb, yb = X[i:i+batch_size], y[i:i+batch_size]
            loss_batch, grad = compute_loss_grad(w, Xb, yb, lam)
            w = opt.update(w, grad)
            total_loss += loss_batch * Xb.shape[0]
        return w, total_loss / n

# ================================================
# Интерактивная функция сравнения
# ================================================
def compare_optimizers(opt_name='Adam', lr=0.01):
    np.random.seed(123)
    w_init = np.random.randn(n_features) * 0.01
    lam = 0.01
    epochs = 80

    opt_classes = {
        'GD': (GDOptimizer, 'full'),
        'SGD': (SGDOptimizer, 1),
        'Momentum': (MomentumOptimizer, 64),
        'Adam': (AdamOptimizer, 64)
    }
    OptClass, batch = opt_classes[opt_name]
    opt = OptClass(lr)
    w = w_init.copy()
    if hasattr(opt, 'v'): opt.v = None
    if hasattr(opt, 'm'): opt.m = None
    if hasattr(opt, 't'): opt.t = 0

    train_losses, val_losses = [], []
    for ep in range(epochs):
        w, train_loss = train_one_epoch(w, X_train, y_train, opt, lam, batch)
        val_loss, _ = compute_loss_grad(w, X_val, y_val, lam)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(range(1, epochs+1), train_losses, label='Train loss')
    ax1.plot(range(1, epochs+1), val_losses, label='Val loss')
    ax1.set_xlabel('Эпоха'); ax1.set_ylabel('Loss')
    ax1.set_yscale('log'); ax1.legend(); ax1.grid(alpha=0.3)
    ax1.set_title(f'{opt_name}, lr={lr}')

    # Финальная accuracy
    prob = sigmoid(X_val @ w)
    pred = (prob >= 0.5).astype(int)
    acc = np.mean(pred == y_val)
    ax2.bar(['Val Accuracy'], [acc], color='steelblue')
    ax2.set_ylim(0, 1); ax2.set_ylabel('Accuracy')
    ax2.text(0, acc + 0.02, f'{acc:.3f}', ha='center', fontsize=12)
    ax2.set_title(f'Точность: {acc:.3f}')
    plt.tight_layout()
    plt.show()

interact(compare_optimizers,
         opt_name=Dropdown(options=['GD', 'SGD', 'Momentum', 'Adam'], value='Adam', description='Оптимизатор'),
         lr=FloatLogSlider(value=0.01, base=10, min=-3, max=0, step=0.1, description='Learning rate'));

График демонстрирует характерные черты каждого метода. GD плавно, но медленно приближается к минимуму. SGD сильно осциллирует, но может быстро снизить потери. Momentum сглаживает колебания и ускоряет продвижение. Adam достигает наименьшего значения за считанные эпохи. Однако при неудачном шаге Adam может «застрять» в локальной области, тогда как SGD продолжит медленное, но устойчивое исследование.

Обратите внимание: попробуйте для Adam `lr = 0.001` и `lr = 0.1`. При малом шаге обучение замедляется, но остаётся стабильным; при большом — Adam может осциллировать, но всё ещё работает. Для GD и SGD диапазон рабочих `lr` значительно уже — это иллюстрирует высокую робастность адаптивных методов.

#### Эксперимент 2: SVM с линейным ядром на реальном датасете

На наборе данных `breast_cancer` сравним линейные SVM, обученные разными вариантами SGD, с точным решением через двойственную задачу (SVC). Запустите ячейку ниже.

In [ ]:
# @title Эксперимент 2: SGD варианты через кастомные оптимизаторы vs точный SVM
from sklearn.datasets import load_breast_cancer
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import time

data = load_breast_cancer()
X_raw, y_raw = StandardScaler().fit_transform(data.data), data.target
y = 2 * y_raw - 1  # метки -1, 1 для hinge loss

# Добавляем свободный член
X = np.c_[X_raw, np.ones(X_raw.shape[0])]

# Функция потерь для hinge loss
def hinge_loss_grad(w, Xb, yb, lam):
    margins = yb * (Xb @ w)
    loss = np.mean(np.maximum(0, 1 - margins)) + 0.5 * lam * np.dot(w, w)
    # субградиент
    ind = (margins < 1).astype(float)
    grad = -(Xb.T @ (yb * ind)) / Xb.shape[0] + lam * w
    return loss, grad

# Сравнение с точным SVM
svc = SVC(kernel='linear', C=1/(0.0001 * X.shape[0]), random_state=42)  # α = 1/(2λn) примерно
svc.fit(X_raw, y_raw)
svc_acc = svc.score(X_raw, y_raw)

# Обучение SGD вариантов
np.random.seed(123)
configs = {
    'SGD plain': (SGDOptimizer(lr=0.001), 1),
    'SGD momentum': (MomentumOptimizer(lr=0.001, beta=0.9), 64),
    'SGD Nesterov': (MomentumOptimizer(lr=0.001, beta=0.9), 64),
    'Adam': (AdamOptimizer(lr=0.001), 64)
}

for name, (opt, batch) in configs.items():
    w = np.random.randn(X.shape[1]) * 0.01
    if hasattr(opt, 'v'): opt.v = None
    if hasattr(opt, 'm'): opt.m = None
    if hasattr(opt, 't'): opt.t = 0
    t0 = time.time()
    for ep in range(100):
        w, _ = train_one_epoch(w, X, y, opt, lam=0.0001, batch_size=batch)
    t1 = time.time()
    pred = np.sign(X @ w)
    acc = np.mean(pred == y)
    print(f"{name:15s} | accuracy {acc:.4f} | time {t1-t0:.4f} s")

print(f"\n{'SVC (libsvm)':15s} | accuracy {svc_acc:.4f}")

SGD‑варианты сходятся за несколько десятков проходов и дают точность, близкую к точному SVM. Главное преимущество SGD — масштабируемость: стоимость одной эпохи $O(nd)$, тогда как точное решение QP-задачи требует $O(n^2)$ памяти и $O(n^3)$ времени.


#### Эксперимент 3: ранняя остановка как неявная регуляризация

Ранняя остановка — простой и мощный приём: обучение прекращается, когда валидационная ошибка перестаёт уменьшаться. Это ограничивает эффективное число итераций и не даёт модели подстроиться под шум. Запустите ячейку.

In [ ]:
# @title Эксперимент 3: ранняя остановка
w = np.random.randn(n_features) * 0.01
opt = AdamOptimizer(lr=0.01)
best_val_loss = np.inf
patience, no_improve = 10, 0
train_losses, val_losses = [], []

for ep in range(200):
    w, train_loss = train_one_epoch(w, X_train, y_train, opt, lam=0.01, batch_size=64)
    val_loss, _ = compute_loss_grad(w, X_val, y_val, lam=0.01)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    if val_loss < best_val_loss:
        best_val_loss, no_improve = val_loss, 0
        best_w = w.copy()
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {ep+1}")
            break

plt.figure(figsize=(7,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.axvline(x=len(train_losses)-patience, color='gray', linestyle='--', label='Stopping point')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Early stopping')
plt.show()


#### 4. Связь оптимизации и выбора модели

Инструменты оптимизации и валидации глубоко переплетены. Learning curves (кривые обучения) — графики ошибки на обучении и валидации в зависимости от числа итераций — позволяют диагностировать недо‑ и переобучение.
- Если обе ошибки высоки и близки — модель недообучена (высокое смещение); можно увеличить ёмкость модели или learning rate.
- Если обучающая ошибка продолжает падать, а валидационная растёт — переобучение (высокая дисперсия); помогает ранняя остановка или усиление регуляризации.

**Регуляризация через раннюю остановку.** Для линейной регрессии с квадратичной функцией потерь и градиентным спуском ранняя остановка на $T$ итерациях эквивалентна явной $L_2$-регуляризации с параметром $\lambda \sim 2/(\eta T)$.

---

**Углублённый взгляд: эквивалентность early stopping и L₂-регуляризации**

Рассмотрим линейную регрессию с целевой функцией $f(\mathbf{w}) = \frac{1}{2}\|\mathbf{y} - X\mathbf{w}\|^2$. Градиентный спуск с шагом $\eta$ и начальным $\mathbf{w}_0 = 0$ даёт последовательность
$$
\mathbf{w}_{t} = (I - \eta X^TX)\,\mathbf{w}_{t-1} + \eta X^T\mathbf{y}.
$$
Итерационное разложение приводит к
$$
\mathbf{w}_{T} = \eta \sum_{k=0}^{T-1} (I - \eta X^TX)^k X^T\mathbf{y}.
$$
При малых $\eta$ это можно аппроксимировать интегралом, и результат близок к $ (X^TX + \lambda I)^{-1} X^T\mathbf{y} $ с $\lambda \approx \frac{2}{\eta T}$ (точнее, $\lambda = 2/(\eta T)$ при $\eta \to 0$). Таким образом, остановка на шаге $T$ имитирует гребневую регрессию с эффективным параметром регуляризации, обратно пропорциональным числу итераций. Более строгий анализ показывает, что спектр $ (I-\eta X^TX)^T $ действует как затухающий фильтр собственных значений, идентичный фильтру $ (X^TX + \lambda I)^{-1} $.

---

#### 5. Практические рекомендации и выводы

Сводка эмпирических и теоретических свойств оптимизаторов приведена в таблице.

| Метод      | Память     | Робастность к $\eta$ | Сходимость (теория)        | Обобщение     |
|------------|------------|------------------------|----------------------------|---------------|
| GD         | $O(d)$   | низкая                 | экспоненц. (сильно вып.)   | среднее       |
| SGD        | $O(d)$   | низкая                 | $O(1/\sqrt{t})$          | хорошее       |
| Momentum   | $O(d)$   | средняя                | экспоненц. (улучш. конст.) | среднее       |
| Adam       | $O(2d)$  | высокая                | не гарантирована (невып.)  | может уступать|

**Рекомендации.**
- Для выпуклых задач с умеренным $n$ — GD или Momentum с ручным подбором шага.
- Для больших разреженных данных — AdaGrad.
- Для глубокого обучения — Adam (быстрый старт) с последующей тонкой настройкой SGD с моментом, если критично предельное обобщение.
- Всегда используйте кросс‑валидацию для выбора learning rate и параметров регуляризации.
- Контролируйте learning curves; ранняя остановка — простой и мощный регуляризатор, особенно в сочетании с адаптивными оптимизаторами.

Оптимизация не существует в вакууме — её настройки непосредственно влияют на итоговое качество модели. Понимание связи между скоростью сходимости, робастностью к гиперпараметрам и обобщающей способностью позволяет осознанно выбирать оптимизатор под конкретную задачу.

---

### Вопросы для самопроверки

**Для начинающих**
1. Почему ранняя остановка действует как регуляризатор? При каком поведении валидационной кривой её применяют?
2. Что делает Adam более робастным к выбору learning rate по сравнению с SGD?
3. Как по графикам learning curves распознать переобучение и недообучение?
4. Почему SGD с моментом иногда даёт лучшее обобщение, чем Adam?

**Для профессионалов**
1. Докажите эквивалентность ранней остановки градиентного спуска и $L_2$-регуляризации для линейной регрессии. Оцените эффективный параметр регуляризации.
2. Сравните вычислительную сложность Adam и SGD с моментом с точки зрения операций на итерацию и использования памяти.
3. Проанализируйте кривые сходимости GD, SGD и Adam на плохо обусловленной квадратичной функции. Почему GD с постоянным шагом может сходиться медленно, а Adam — быстро?
4. Опишите условия, при которых Adam может расходиться (контрпример Reddi et al.). Предложите модификацию, гарантирующую сходимость.
5. Выведите критерий ранней остановки, основанный на дисперсии стохастического градиента. Как он связан с оптимальным числом итераций?

### 9.5. Оптимизация SVM: SMO, Pegasos и связь с двойственностью

Метод опорных векторов — один из немногих классических алгоритмов, для которого существует как прямая, так и двойственная формулировка задачи оптимизации. Выбор между ними диктуется размером выборки, типом ядра и доступной памятью. В этом разделе мы детально рассмотрим два основных подхода: SMO, решающий двойственную задачу и позволяющий применять ядровой трюк, и Pegasos — масштабируемый субградиентный метод для прямой задачи.

---

#### 1. Особенности задачи SVM с точки зрения оптимизации

**Прямая задача (линейное ядро, soft‑margin).** Пусть имеется выборка $\{\mathbf{x}_i, y_i\}_{i=1}^n$, $y_i \in \{-1, +1\}$. Прямая задача SVM с квадратичной регуляризацией записывается как

$$
\min_{\mathbf{w}} \; \frac{1}{n}\sum_{i=1}^n \max\bigl(0, 1 - y_i \mathbf{w}^T \mathbf{x}_i\bigr) + \frac{\lambda}{2}\|\mathbf{w}\|^2.
$$

Функция потерь $\ell(y, f) = \max(0, 1 - y f)$ (hinge loss) не дифференцируема в точке перегиба, поэтому требуются субградиентные методы. Задача остаётся выпуклой, но не строго выпуклой при $\lambda=0$ и вырожденных данных, что может приводить к нескольким оптимальным решениям.

**Двойственная задача (soft‑margin с ядром).** Вводя множители Лагранжа $\boldsymbol{\alpha} \ge 0$, получаем двойственную задачу для SVM с допустимым порогом $C = 1/(2\lambda n)$ (в нотации sklearn):

$$
\begin{aligned}
\max_{\boldsymbol{\alpha}} \quad & \sum_{i=1}^n \alpha_i - \frac12 \sum_{i,j=1}^n \alpha_i \alpha_j y_i y_j K(\mathbf{x}_i, \mathbf{x}_j) \\
\text{s.t.} \quad & 0 \le \alpha_i \le C, \quad \sum_{i=1}^n \alpha_i y_i = 0.
\end{aligned}
$$

Здесь $K(\mathbf{x}_i, \mathbf{x}_j) = \langle \phi(\mathbf{x}_i), \phi(\mathbf{x}_j) \rangle$ — функция ядра. Матрица $Q_{ij} = y_i y_j K_{ij}$ является положительно полуопределённой, плотной в общем случае и требует $O(n^2)$ памяти. Решение двойственной задачи даёт разрежённый вектор $\boldsymbol{\alpha}$: только опорные векторы имеют $\alpha_i > 0$.

Основная дилемма: двойственная задача позволяет использовать ядровой трюк без явного перехода в пространство признаков, но цена — квадратичная память и вычислительная сложность. Прямая задача работает в исходном пространстве признаков (не позволяя нелинейные ядра без преобразования), однако допускает линейное по $n$ масштабирование при использовании стохастических методов.

---

#### 2. SMO (Sequential Minimal Optimization) — стандарт для SVM

Алгоритм SMO решает двойственную задачу, оптимизируя за одну итерацию только два множителя $\alpha_i$ и $\alpha_j$. Это возможно благодаря линейному ограничению $\sum \alpha_k y_k = 0$: изменение двух переменных позволяет сохранить равенство.

**Аналитическое обновление пары.** Зафиксируем все $\alpha_k$, кроме $\alpha_i$ и $\alpha_j$. Из ограничения $\alpha_i y_i + \alpha_j y_j = \text{const}$ выразим $\alpha_i = y_i(\text{const} - \alpha_j y_j)$. Подставляя в двойственную целевую функцию, получаем квадратичную функцию от $\alpha_j$:

$$
\Psi(\alpha_j) = \frac12 \eta \alpha_j^2 + (y_j(E_i - E_j) - \eta \alpha_j^{\text{old}})\alpha_j + \text{const},
$$

где $E_k = f(\mathbf{x}_k) - y_k$ — ошибка на $k$-м объекте ($f(\mathbf{x}_k) = \sum_{t} \alpha_t y_t K_{tk} + b$), $\eta = K_{ii} + K_{jj} - 2K_{ij}$. Максимум без учёта ограничений достигается в точке

$$
\alpha_j^{\text{new, unc}} = \alpha_j^{\text{old}} + \frac{y_j (E_i - E_j)}{\eta}.
$$

Далее решение клиппируется в допустимый интервал $[L, H]$, определяемый границами $0 \le \alpha \le C$ и знаками $y_i, y_j$. После клиппинга обновляется $\alpha_i$ из линейного ограничения и смещение $b$.

**Эвристики выбора пары.** Первый множитель выбирается среди объектов, нарушающих условия ККТ (например, $\alpha_i=0$, но $y_i f(\mathbf{x}_i) < 1$; $\alpha_i=C$, но $y_i f(\mathbf{x}_i) > 1$; $0<\alpha_i<C$, но $y_i f(\mathbf{x}_i) \ne 1$). Второй множитель подбирается так, чтобы максимизировать $|E_i - E_j|$, что ускоряет сходимость.

**Сходимость.** Доказано, что SMO сходит за конечное число итераций при использовании правила выбора пары, гарантирующего строгое возрастание целевой функции. На практике сложность составляет от $O(n)$ до $O(n^2)$ в зависимости от данных. Кеширование матрицы $K$ позволяет избежать повторных вычислений, но требует $O(n^2)$ памяти. Это ограничивает применимость SMO выборками до $\sim 10^4$–$10^5$ объектов.

---

#### 3. Pegasos (Primal Estimated sub‑GrAdient SOlver)

Для больших выборок с линейным ядром эффективен прямой субградиентный метод Pegasos. Он минимизирует регуляризованный эмпирический риск

$$
f(\mathbf{w}) = \frac{\lambda}{2}\|\mathbf{w}\|^2 + \frac{1}{n}\sum_{i=1}^n \max(0, 1 - y_i \mathbf{w}^T \mathbf{x}_i).
$$

На итерации $t$ выбирается случайное подмножество $A$ размера $k$ (обычно $k=1$), и вычисляется субградиент:

$$
\nabla_t = \lambda \mathbf{w}_t - \frac{1}{|A|}\sum_{i \in A, \; y_i \mathbf{w}_t^T \mathbf{x}_i < 1} y_i \mathbf{x}_i.
$$

Обновление параметров:

$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \eta_t \nabla_t,
$$

где $\eta_t = 1/(\lambda t)$ — шаг, гарантирующий сходимость. Эквивалентно,

$$
\mathbf{w}_{t+1} = (1 - \eta_t\lambda)\mathbf{w}_t + \frac{\eta_t}{|A|}\sum_{i \in A, \; y_i \mathbf{w}_t^T \mathbf{x}_i < 1} y_i \mathbf{x}_i.
$$

**Теоретическая гарантия.** Для выпуклой $L$-липшицевой функции потерь Pegasos достигает точности $\epsilon$ за $O(1/(\lambda \epsilon))$ итераций, что оптимально для субградиентных методов. Каждая итерация занимает $O(d|A|)$ времени, метод масштабируется на миллионы объектов.

**Недостаток:** работает только с линейным ядром. Однако в сочетании с техникой случайных признаков (Random Fourier Features) или аппроксимацией Нюстрёма его можно расширить на нелинейные ядра.

---

**Углублённый взгляд: связь Pegasos с SGD и проекцией на шар**

Рассмотрим замену переменных $\mathbf{w} = \frac{1}{\sqrt{\lambda}} \mathbf{v}$. Тогда прямая задача сводится к минимизации по $\mathbf{v}$ функции $\frac{1}{2}\|\mathbf{v}\|^2 + \frac{1}{\lambda n}\sum \max(0, \sqrt{\lambda} - y_i \mathbf{v}^T \mathbf{x}_i)$. Субградиентное обновление без регуляризации эквивалентно SGD с последующим масштабированием. Более того, можно показать, что Pegasos с шагом $\eta_t = 1/(\lambda t)$ и проекцией $\mathbf{w}$ на шар радиуса $R = 1/\sqrt{\lambda}$ даёт ту же итерацию. Действительно, из условия $\|\mathbf{w}^*\| \le 1/\sqrt{\lambda}$ (оптимальное решение лежит в шаре) проецирование на этот шар ускоряет сходимость и стабилизирует обучение.

---

#### 4. Сравнение SMO и Pegasos на практике

Проведём эксперимент на синтетических данных, варьируя размер выборки $n$. Сравним `SVC(kernel='linear')` (SMO, двойственное решение), `LinearSVC` (liblinear, прямая задача с координатным спуском) и `SGDClassifier(loss='hinge')` (Pegasos‑подобный SGD).



In [ ]:
# @title Сравнение SMO, LinearSVC и Pegasos (интерактивный)
import numpy as np
import time
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from ipywidgets import interact, IntSlider

def compare_svm_solvers(n_samples=2000):
    n_features = 100
    X, y = make_classification(n_samples=n_samples, n_features=n_features,
                               n_informative=50, n_redundant=20, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    # Конфигурация методов: (имя, класс, параметры)
    methods = [
        ('SVC (SMO)', SVC, {'kernel': 'linear', 'C': 1.0, 'random_state': 42}),
        ('LinearSVC', LinearSVC, {'C': 1.0, 'random_state': 42, 'max_iter': 5000, 'dual': False}),
        ('SGD (Pegasos)', SGDClassifier, {'loss': 'hinge', 'penalty': 'l2',
                          'alpha': 1.0/(2*n_samples), 'random_state': 42,
                          'max_iter': 1000, 'tol': 1e-4})
    ]

    print(f"Размер выборки: {n_samples}")
    print(f"{'Метод':<15} {'Accuracy':<10} {'Время (с)':<10} {'#SV'}")
    print('-' * 45)

    for name, Model, params in methods:
        model = Model(**params)
        t0 = time.time()
        model.fit(X_train, y_train)
        elapsed = time.time() - t0
        acc = accuracy_score(y_test, model.predict(X_test))
        n_sv = len(model.support_) if hasattr(model, 'support_') else '—'
        print(f"{name:<15} {acc:.4f}      {elapsed:.4f}      {n_sv}")

interact(compare_svm_solvers,
         n_samples=IntSlider(value=2000, min=500, max=10000, step=500,
                             description='Размер выборки'));



Результат (типичный вывод) показывает, что при $n=1000$ все методы дают сравнимую точность, но SMO может быть медленнее из‑за решения двойственной задачи. `LinearSVC` демонстрирует наилучший баланс скорости и точности для средних $n$. При $n=20000$ SMO всё ещё работает, но время растёт квадратично; `LinearSVC` остаётся быстрым; `SGDClassifier` обгоняет всех по скорости, незначительно уступая в точности. Число опорных векторов у SMO позволяет судить о разрежённости решения.

**Выводы:**
- До $n \sim 10^4$ SMO даёт точное решение, удобное для ядровых методов.
- Для линейного ядра и $n > 10^5$ Pegasos/SGD — единственный масштабируемый выбор.
- `LinearSVC` (liblinear) — отличный компромисс для линейных SVM среднего размера.

---

#### 5. Связь с гиперпараметрами и валидацией

Выбор $C$ и параметров ядра через кросс‑валидацию требует многократного обучения SVM. SMO с кешированием матрицы $K$ имеет преимущество: при изменении $C$ (но не $\gamma$) матрица $K$ не меняется, и можно повторно использовать её, ускоряя поиск. В реализации sklearn (`SVC`) кеш хранится вплоть до нескольких гигабайт, что на практике ограничивает $n$.

Для линейного SVM ранняя остановка SGD‑классификатора (Pegasos) эквивалентна $L_2$-регуляризации с $\lambda \approx 1/(\eta T)$, что позволяет контролировать эффективную сложность модели без повторного обучения.

**Практические рекомендации:**
- Линейное ядро, $n > 10^5$: `SGDClassifier(loss='hinge')` или `LinearSVC`.
- Линейное ядро, $n < 10^4$: `SVC(kernel='linear')` или `LinearSVC` — точное решение.
- Нелинейные ядра (RBF, полиномиальное), $n \sim 10^4$: `SVC` с кешированием.
- Нелинейные ядра с $n > 10^5$: аппроксимация ядра (Nyström, Random Fourier Features) + SGD.

Понимание этих компромиссов позволяет осознанно выбирать оптимизационный алгоритм для SVM, сочетая точность и вычислительную эффективность.

---

### Вопросы для самопроверки

**Для начинающих**
1. Почему в SMO оптимизируются ровно два множителя за итерацию? Какое ограничение это позволяет удовлетворить?
2. Запишите аналитическую формулу обновления $\alpha_j$ без учёта границ. Откуда берётся величина $\eta$?
3. В чём преимущество Pegasos перед SMO при большом числе объектов? Какова цена одной итерации Pegasos?
4. Почему SMO позволяет использовать ядровой трюк, а Pegasos — только линейные ядра (без специальных ухищрений)?

**Для профессионалов**
1. Выведите формулу обновления SMO для $\alpha_j^{\text{new, unc}}$, минимизируя квадратичную функцию по $\alpha_j$ при фиксированных остальных переменных.
2. Докажите, что Pegasos с шагом $\eta_t = 1/(\lambda t)$ сходится со скоростью $O(1/(\lambda T))$ для ожидаемой ошибки. Какие предположения о данных требуются?
3. Сравните общее число операций (время) SMO и Pegasos в терминах $n$ и $d$. При каком $n$ SMO становится неприменимым из‑за квадратичной памяти?
4. Покажите эквивалентность обновления Pegasos и SGD с последующей проекцией на шар радиуса $1/\sqrt{\lambda}$. Как эта проекция помогает ускорить сходимость?
5. Предложите способ масштабирования SMO на $n \sim 10^6$ с использованием аппроксимации матрицы ядра. Как изменится задача и гарантии сходимости?